# Data&Results
- This is the notebook used to download inspect and preprocess data
- This notebook implements the comparison between **Gemma-3N-E2B-it** and **Qwen2.5-Omni-3B** on two modalities: **Text** and **Audio**.
- This notebook implements the comparison between the Gemma fine-tuned models across different input modalities





## 1. Setup & Dependencies
First, we install the necessary libraries. Note that we need `edge-tts` for generating audio from the text datasets.

In [1]:
!pip install -q edge-tts nest_asyncio

In [2]:
import torch
import transformers
from datasets import load_dataset
import os
import random
import pandas as pd
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import seaborn as sns
import asyncio
import edge_tts
from tqdm import tqdm
import librosa
import numpy as np
import json
from google.colab import drive
drive.mount('/content/drive')

random.seed(42)

DOWNLOAD = False
CLEAN = False
DRIVE_DIR = "/content/drive/MyDrive/TextMiningProject/"
DATASET = os.path.join(DRIVE_DIR, "data")
os.makedirs(DATASET, exist_ok=True)

Mounted at /content/drive


In [3]:
from huggingface_hub import login
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
    login(token)
    print("Logged in via Colab Secret")
except:
    login()

Logged in via Colab Secret


## 2. Data Preparation
We load the **XSum** and **SAMSum** test datasets and select a random subset of 300 samples each to keep the runtime manageable, this is for inference.

#### Test

In [4]:
NUM_SAMPLES = 300 # for 600 documents in total
ALL_DATA_JSON = os.path.join(DATASET, "all_data.json")

if DOWNLOAD:
    def prepare_dataset(dataset_name, split, text_column, summary_column, num_samples):
        print(f"Loading {dataset_name}...")
        ds = load_dataset(dataset_name, split=split)
        ds = ds.shuffle(seed=42).select(range(min(num_samples, len(ds))))

        data = []
        for idx, item in enumerate(ds):
            entry = {
                "id": f"{dataset_name.split('/')[-1]}_{idx}",
                "text": item[text_column],
                "summary": item[summary_column],
                "dataset": f"{dataset_name.split('/')[-1]}"
            }
            data.append(entry)
        return data

        samsum_data = prepare_dataset("knkarthick/samsum", "test", "dialogue", "summary", NUM_SAMPLES)
        xsum_data = prepare_dataset("EdinburghNLP/xsum", "test", "document", "summary", NUM_SAMPLES)
        all_data = xsum_data + samsum_data
        print(f"Total samples: {len(all_data)}")

    samsum_data = prepare_dataset("knkarthick/samsum", "test", "dialogue", "summary", NUM_SAMPLES)
    xsum_data = prepare_dataset("EdinburghNLP/xsum", "test", "document", "summary", NUM_SAMPLES)
    all_data = xsum_data + samsum_data

    with open(ALL_DATA_JSON, "w", encoding="utf-8") as f:
        json.dump(all_data, f, ensure_ascii=False, indent=2)
    print(f"Saved all_data to {ALL_DATA_JSON}")
else:
    with open(ALL_DATA_JSON, "r", encoding="utf-8") as f:
        all_data = json.load(f)
        samsum_data = [item for item in all_data if item["dataset"] == "samsum"]
        xsum_data   = [item for item in all_data if item["dataset"] == "xsum"]
        print(f"Loaded all_data from {ALL_DATA_JSON}")

Loaded all_data from /content/drive/MyDrive/TextMiningProject/data/all_data.json


In [ ]:
print("The data used for training contains:", len(all_data), "datapoints in total.")
print("Samsum contains", len(samsum_data), "datapoints.")
print("Xsum contains", len(xsum_data), "datapoints.")

The data used for training contains: 600 datapoints in total.
Samsum contains 300 datapoints.
Xsum contains 300 datapoints.


#### Train Data for fine-tuning gemma

In [5]:
NUM_SAMPLES = 450
ALL_DATA_JSON = os.path.join(DATASET, "all_data_train.json")

if DOWNLOAD:
    def prepare_dataset(dataset_name, split, text_column, summary_column, num_samples):
        print(f"Loading {dataset_name}...")
        ds = load_dataset(dataset_name, split=split)
        ds = ds.shuffle(seed=42).select(range(min(num_samples, len(ds))))

        data = []
        for idx, item in enumerate(ds):
            entry = {
                "id": f"{dataset_name.split('/')[-1]}_{idx}",
                "text": item[text_column],
                "summary": item[summary_column],
                "dataset": f"{dataset_name.split('/')[-1]}"
            }
            data.append(entry)
        return data

        samsum_data = prepare_dataset("knkarthick/samsum", "train", "dialogue", "summary", NUM_SAMPLES)
        xsum_data = prepare_dataset("EdinburghNLP/xsum", "train", "document", "summary", NUM_SAMPLES)
        all_data = xsum_data + samsum_data
        print(f"Total samples: {len(all_data)}")

    samsum_data_train = prepare_dataset("knkarthick/samsum", "train", "dialogue", "summary", NUM_SAMPLES)
    xsum_data_train = prepare_dataset("EdinburghNLP/xsum", "train", "document", "summary", NUM_SAMPLES)
    all_data_train = xsum_data_train + samsum_data_train

    with open(ALL_DATA_JSON, "w", encoding="utf-8") as f:
        json.dump(all_data_train, f, ensure_ascii=False, indent=2)
    print(f"Saved all_data to {ALL_DATA_JSON}")
else:
    with open(ALL_DATA_JSON, "r", encoding="utf-8") as f:
        all_data_train = json.load(f)
        samsum_data_train = [item for item in all_data if item["dataset"] == "samsum"]
        xsum_data_train   = [item for item in all_data if item["dataset"] == "xsum"]
        print(f"Loaded all_data from {ALL_DATA_JSON}")

Loaded all_data from /content/drive/MyDrive/TextMiningProject/data/all_data_train.json


In [ ]:
print("The data used for training contains:", len(all_data_train), "datapoints in total.")
print("Samsum contains", len(samsum_data_train), "datapoints.")
print("Xsum contains", len(xsum_data_train), "datapoints.")

The data used for training contains: 900 datapoints in total.
Samsum contains 450 datapoints.
Xsum contains 450 datapoints.


#### Display

In [ ]:
def show_random_samples(dataset_list, num_examples=3):
    if len(dataset_list) < num_examples:
        num_examples = len(dataset_list)

    picks = random.sample(range(len(dataset_list)), num_examples)

    selected_data = []
    for i in picks:
        item = dataset_list[i].copy()
        if len(item['text']) > 1000:
            item['text'] = item['text'][:1000] + "..."
        selected_data.append(item)

    df = pd.DataFrame(selected_data)
    cols = ['id', 'text', 'summary', 'dataset']
    df = df[[c for c in cols if c in df.columns]]

    display(HTML(df.to_html(index=False)))

In [ ]:
print("--- XSum Samples ---")
show_random_samples(xsum_data)
print("\n--- SAMSum Samples ---")
show_random_samples(samsum_data)

--- XSum Samples ---


id,text,summary,dataset
xsum_57,"Spanish police say he had secretly taken photographs of his students as well as having sent images of other children to people on the internet.\nThe Manchester-born man reportedly had a ""special obsession"" with a girl he taught.\nHe was arrested in Valladolid, north-west Spain as part of an Interpol investigation.\nInterpol said they discovered illegal files were being sent from an email address in Spain and alerted the authorities.\nPolice said on Sunday the teacher is suspected of using a cloud storage service to keep and share the pictures between computers and with others.\nOfficers say they found a large number of sexually explicit files involving minors at the home of the teacher.\nHe kept photos of one girl in a dedicated folder on his computer but had not shared the images he had secretly taken of his underage students.\nPolice said they believe he previously lived in the Seville area.\nSpanish National Police said: ""At the moment, [police are] trying to establish the identity of the v...",A British teacher living in Spain has been arrested for allegedly storing and sharing sexual images of children.,xsum
xsum_12,"According to three separate analyses, a flood of automated comments to the Federal Communications Commission (FCC) was detected over the weekend.\nMore than 400,000 comments with remarkably similar wording have been detected in recent days.\nNet neutrality proponents argue that all internet traffic should be equal.\nThis means that no content provider should be able to, for example, charge more for faster access to certain data.\nOne expert described bot activity as a new form of protest.\n""Someone has gone out of their way to make these seem like real submissions,"" wrote Chris Sinchok in a blog post about the apparently automated activity.\nHaving downloaded the comments and associated data, Mr Sinchok noticed that the names and email addresses associated with thousands of them also turned up in lists of personal data stolen from websites.\nHe told the BBC that this suggested someone might be using information collected from breached databases to make the submissions look more authentic.\n""It...","Bots appear to be spamming a US regulator's website over a proposed reversal of net neutrality rules, researchers have said.",xsum
xsum_140,"The event represents the final opportunity for GB to attain a maximum of two berths (one male and one female) for August's Rio Olympics.\nThe UK Sport-funded GB women's team will need to rank in the top six from nations who have not already qualified.\n""We'll give it our all to qualify for the Olympics in Rio this summer,"" Smith told BBC Sport.\n""It's going to be a tough competition but I'm confident in my preparations.""\nCommonwealth champion Smith is the lead candidate to represent Team GB should the GB women claim a berth, but rising star Rebekah Tiler is expected to rival her.\nEx-England lifter Hannah Powell - who now aims to represent Wales - will line up at her first major international since the 2011 World Championships.\nThe unfunded British men's team need a top-seven place at the European Championships in Forde, which run from 10 to 16 April.\nThey are boosted by Welsh lifter Darius Jokarzadeh's first appearance in the sport since finishing fourth at the 2014 Commonwealth Games.\nTh...",London Olympian Zoe Smith will lead a 14-strong British team at April's European Championships in Norway.,xsum



--- SAMSum Samples ---


id,text,summary,dataset
samsum_125,"Rob: I wanna start Get off the couch challenge! Who's with me? I'm gonna start with 30 min of activity every day.\nGreg: way ahead of you mate! at least 60 min a day!\nRob: good for you! 30 min a day is gonna be challenging for me!\nGreg: good luck then!\nWill: don't be too hard on yourself. better to be realistic. fingers crossed!\nAnna: i'm in! 30 min sounds reasonable! \nGreg: look for variety of exercises such as walking, biking, swimming whatever you enjoy!\nAnna: i'm gonna need lots of support from you guys!",Rob wants to start doing 30 minutes of physical activity every day. Greg is already doing 60 minutes. Anna wants to start exercising just like Rob.,samsum
samsum_114,"Sam: Where are you?\nKate: downstairs\nSam: already?\nKate: sure, come down\nJeff: We're in the little room next to the reception\nKate: have you noticed the woman making a cake?\nSam: yes\nKate: did you see the cake?\nSam: I didn't pay attention, why?\nKate: there was a huge penis on it\nKate: made of marzipan or sth\nSam: hahah, really?\nKate: when I noticed it, I started laughing\nKate: so she laughed too\nSam: nice\nKate: I asked her what's she doing\nKate: Somebody ordered a cake with a huge penis with balls and chocolate hair\nSam: hahah, I must see it!\nKate: she is working in the small room under the stairs\nSam: right, there is a kitchen",Kate and Jeff are downstairs in a room next to the reception. Some lady is making a marzipan cake.,samsum
samsum_71,"Steffen: Any room in any of the cars going to the infinity pool? Im more handicapped than usual since I twisted my ancle yesterday :(\nIrene: we can give you a lift. Don’t think the car can make it all the way up, so will park at the bottom and hike up \nSteffen: Then I think I have to skip - cant really walk on my leg atm :confused: But thanks anyway\nIrene: :(\nDan: I’m pretty sure Mr.Budd could make it, it’s 4wheel drive, if mr.budd is going, although I haven’t seen the hill \nLuke: have you been up there? how bad is the road actually?\nLuke: lol, that explains it\nLuke: Sandy, is it vistas de olas?\nBen: Yes! Vistas de olas","Steffen twisted his ankle yesterday and needs a lift to the infinity pool. Irene's car probably won't make it up the hill, so they'd have to park at the bottom and hike up. Mr.Budd should make it up the hill since it's a 4-wheel drive.",samsum


In [ ]:
print("--- XSum Samples ---")
show_random_samples(xsum_data_train)
print("\n--- SAMSum Samples ---")
show_random_samples(samsum_data_train)

--- XSum Samples ---


id,text,summary,dataset
xsum_327,"Yet even he must regret the rash comments that have seen him declared a ""persona non grata"" by the organisers of the Cannes Film Festival in France.\nIt was apparently clear to those present at Wednesday's press conference that he was joking when he declared himself a Nazi who felt sympathy for Adolf Hitler.\nIt was also clear, though, that his misguided attempts at humour had taken him into areas where there is little humour to be found.\nBorn in 1956 in Copenhagen, Lars Trier began making movies as a child with a Super 8 camera.\nHe went on to study at the Danish Film School, where he was encouraged by his fellow students to adopt the ""Von"".\nAward-winning student films were followed by his first feature, The Element of Crime, in 1984.\nA nightmarish, visually distinctive thriller, it became the first of several Von Trier works to be shortlisted for Cannes' prestigious Palme d'Or award.\nThe director was back in Cannes in 1991 with Europa, a drama set in Germany in the aftermath of World Wa...","The Danish film-maker Lars Von Trier has never shied away from controversial material, or from making contentious pronouncements.",xsum
xsum_57,"In July, UK Energy Secretary Greg Clark approved SP Manweb's proposal for 17km (10.5 miles) of power lines linking Clocaenog wind farm to a substation at Glascoed.\nThis followed a public inquiry into the plans last year.\nSP Manweb said the decision to have a review does not change their programme.\nThe High Court hearing will take place in Llangefni in April.\nThe Pylon the Pressure group are campaigning for the cables, which will carry supplies from four windfarms in the Clocaenog and Brenig areas, to be laid underground.\nThe group's chairman, Dyfrig Hughes, said the scheme will ""blight one of the most beautiful and historic landscapes of north Wales"".\nHe added: ""Unfortunately for us, the UK government agreed with them [SP Manweb] and granted permission despite underground cabling costing no more than overhead lines over the lifetime of the connection.""\nDuring the inquiry, then UK Energy Secretary Ed Davey said the additional Â£16m cost to lay the cables underground would be disproporti...",Opponents of an overhead cables scheme across parts of rural Denbighshire and Conwy have won the right to a judicial review.,xsum
xsum_12,"Mark Mercer, 47, of Toll Bar Houses, Workington, is accused of taking ??1,530 from Maryport Post Office on Monday as well as several firearms offences.\nNo pleas were entered during a brief hearing at Carlisle Magistrates' Court and Mr Mercer was remanded in custody.\nA 24-year-old woman also arrested in connection with the raid has been released on bail until 14 March.",A man has appeared in court charged in connection with an armed robbery at a post office in Cumbria.,xsum



--- SAMSum Samples ---


id,text,summary,dataset
samsum_379,Ron: will be late\nHilary: ok \nRon: sorry for that!,Ron will be late.,samsum
samsum_140,"David: We gotta split the bills guys\nMonica: Sure, I'm looking at this email too\nDavid: How bout you Nina?\nNina: I'm here, I'm here, 3 ways? \nDavid: I was gone for 2 weeks so maybe lets try to divide electricity different ways? \nNina: Sure, any idea how to divide it?\nDavid: I'll make some excel calculations\nMonica: Just send it over, I'm fine with it","David, Monica and Nina will split the bills. David will divide the electricity bill differently, because he was gone for 2 weeks.",samsum
samsum_125,"David: Hi Terry , How’s the work progress,\nTerry: Sorry for the delay boss in posting my work, Its been a rough week, one I have finally caught upto.\nDavid: Well don’t take too long I need the work finished by now, its approaching the deadline.\nTerry: Ok. I will submit right away.","Terry's having a rough week and there's some delay in his work. David, his boss, needs it finished. Terry will submit it now.",samsum


### 2.2 Text Explanatory Data Analysis

- token

In [ ]:
from transformers import AutoTokenizer

tokenizer_qwen = AutoTokenizer.from_pretrained("google/gemma-3n-E2B-it")
tokenizer_gemma = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Omni-3B")


config.json:   0%|          | 0.00/4.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/769 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.63k [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

In [ ]:
def add_token_lengths(df, tokenizer, prefix):
    df[f'text_tokens_{prefix}'] = df['text'].apply(
        lambda x: len(tokenizer(str(x), add_special_tokens=False).input_ids)
    )
    df[f'summary_tokens_{prefix}'] = df['summary'].apply(
        lambda x: len(tokenizer(str(x), add_special_tokens=False).input_ids)
    )
    return df

df = pd.DataFrame(all_data)

df = add_token_lengths(df, tokenizer_qwen, "qwen")
df = add_token_lengths(df, tokenizer_gemma, "gemma")

stats_qwen = df.groupby('dataset')[['text_tokens_qwen', 'summary_tokens_qwen']].agg(['min', 'max', 'mean'])
stats_gemma = df.groupby('dataset')[['text_tokens_gemma', 'summary_tokens_gemma']].agg(['min', 'max', 'mean'])

print("=== Qwen ===")
display(stats_qwen)

print("=== Gemma ===")
display(stats_gemma)


=== Qwen ===


text_tokens_qwen                   summary_tokens_qwen               
                     min   max        mean                 min max       mean
dataset                                                                      
samsum                17   719  160.223333                   5  78  25.283333
xsum                  15  2119  498.513333                   7  61  26.340000

=== Gemma ===


text_tokens_gemma                   summary_tokens_gemma      \
                      min   max        mean                  min max   
dataset                                                                
samsum                 18   700  154.703333                    5  81   
xsum                   15  2096  484.476667                    7  65   

                    
              mean  
dataset             
samsum   25.393333  
xsum     26.786667

- words

In [ ]:
def add_word_lengths(df):
    df['text_words'] = df['text'].apply(lambda x: len(str(x).split()))
    df['summary_words'] = df['summary'].apply(lambda x: len(str(x).split()))
    return df

df = add_word_lengths(df)

stats_words = df.groupby('dataset')[['text_words', 'summary_words']].agg(['min', 'max', 'mean'])

print("=== WORDS ===")
display(stats_words)


=== WORDS ===


text_words                   summary_words               
               min   max        mean           min max       mean
dataset                                                          
samsum           9   516  103.593333             3  54  19.880000
xsum            13  1683  378.450000             4  45  21.093333

- This results show that the number of tokens and words for the summaries of the two dataset is similar. And it's about 20 words/ 25 tokens.
- So is sufficient to use a prompt that asks the model a summary which is long about a sentence for both the dataset. This could be the best to align the generations of the model to the golden standard.
- The result of the number of words in the texts and summaries are comparable to the number of tokens that the tokenizers produce.
- As the tokens and words of the text of the two dataset, this is not comparable as "xsum" contains text four times bigger.

### Audio Generation
Now we generate audio files for each document using `edge-tts`.
- It's used a cut on the character of 500, to not have too long audios and a fair comparison between the two datasets.

In [ ]:
async def generate_audio(text, output_path):
    voice = "en-US-AriaNeural"
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_path)

async def process_batch(data_items):
    semaphore = asyncio.Semaphore(CONCURRENT_TASKS)

    async def sem_generate(item, output_path):
        async with semaphore:
            await generate_audio(item['text'][:500], output_path)

    tasks = []
    for item in data_items:
        file_path = os.path.join(OUTPUT_DIR, f"{item['id']}.wav")
        item['audio_path'] = file_path
        if not os.path.exists(file_path):
            tasks.append(sem_generate(item, file_path))

    if tasks:
        await asyncio.gather(*tasks)

Test Data

In [ ]:
import nest_asyncio

OUTPUT_DIR = os.path.join(DRIVE_DIR, "data", "audio")
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 40
CONCURRENT_TASKS = 8
nest_asyncio.apply()

async def main():
    for i in tqdm(range(0, len(all_data), BATCH_SIZE), desc="Generating Audio"):
        batch = all_data[i:i+BATCH_SIZE]
        await process_batch(batch)

if DOWNLOAD:
    asyncio.run(main())
else:
    for item in all_data:
        file_path = os.path.join(OUTPUT_DIR, f"{item['id']}.wav")
        if os.path.exists(file_path):
            item['audio_path'] = file_path
        else:
            raise FileNotFoundError(f"Audio file for {item['id']} not found in {OUTPUT_DIR}")

Train Data

In [ ]:
import nest_asyncio

OUTPUT_DIR = os.path.join(DRIVE_DIR, "data", "audio_train")
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 40
CONCURRENT_TASKS = 8
nest_asyncio.apply()

async def main():
    for i in tqdm(range(0, len(all_data_train), BATCH_SIZE), desc="Generating Audio"):
        batch = all_data_train[i:i+BATCH_SIZE]
        await process_batch(batch)

if DOWNLOAD:
    asyncio.run(main())
else:
    for item in all_data_train:
        file_path = os.path.join(OUTPUT_DIR, f"{item['id']}.wav")
        if os.path.exists(file_path):
            item['audio_path'] = file_path
        else:
            raise FileNotFoundError(f"Audio file for {item['id']} not found in {OUTPUT_DIR}")

Generating Audio: 100%|██████████| 23/23 [03:23<00:00,  8.84s/it]


 Now we look at the features of the generated audio.

### 2.3 Audio Explanatory Data Analysis

In [ ]:
import soundfile as sf

def audio_stats(data):
    records = []

    for item in tqdm(data, desc="Analyzing audio (per file)"):
        path = item.get("audio_path")
        dataset = item.get("dataset")

        audio, sr = sf.read(path)

        duration = len(audio) / sr
        channels = 1 if audio.ndim == 1 else audio.shape[1]

        records.append({
            "dataset": dataset,
            "path": path,
            "sample_rate": sr,
            "channels": channels,
            "duration_sec": duration,
            "dtype": str(audio.dtype),
            "min_value": float(audio.min()),
            "max_value": float(audio.max()),
            "num_samples": len(audio)
        })

    return pd.DataFrame(records)

def audio_stats_by_dataset(audio_files_df):
    return (
        audio_files_df
        .groupby("dataset")
        .agg(
            num_files=("path", "count"),
            avg_duration_sec=("duration_sec", "mean"),
            min_duration_sec=("duration_sec", "min"),
            max_duration_sec=("duration_sec", "max"),
            avg_sample_rate=("sample_rate", "mean"),
            channels=("channels", lambda x: x.mode().iloc[0]),
            dtype=("dtype", lambda x: x.mode().iloc[0]),
            min_amplitude=("min_value", "min"),
            max_amplitude=("max_value", "max"),
        )
        .reset_index()
    )

Test

In [ ]:
audio_files_df = audio_stats(all_data)
audio_dataset_df = audio_stats_by_dataset(audio_files_df)

audio_dataset_df

Analyzing audio (per file): 100%|██████████| 600/600 [00:16<00:00, 37.16it/s]


,dataset,num_files,avg_duration_sec,min_duration_sec,max_duration_sec,avg_sample_rate,channels,dtype,min_amplitude,max_amplitude
0,samsum,300,33.04616,7.224,57.264,24000.0,1,float64,-0.824205,0.868858
1,xsum,300,32.40672,5.832,49.464,24000.0,1,float64,-0.855517,0.884337


Train

In [ ]:
audio_files_df = audio_stats(all_data_train)
audio_dataset_df = audio_stats_by_dataset(audio_files_df)

audio_dataset_df

Analyzing audio (per file): 100%|██████████| 900/900 [00:30<00:00, 29.57it/s]


,dataset,num_files,avg_duration_sec,min_duration_sec,max_duration_sec,avg_sample_rate,channels,dtype,min_amplitude,max_amplitude
0,samsum,450,32.113920,6.912,68.328,24000.0,1,float64,-0.851747,0.865827
1,xsum,450,32.780373,15.840,49.656,24000.0,1,float64,-0.842533,0.976215


### Preprocessing of audio files


Here, a standardized audio preprocessing pipeline is applied to make all files homogeneous before being used by multimodal models: each file is loaded, resampled to 16 kHz, converted to mono and to `float32`, and then saved to disk in a dedicated directory. This aligns audio inputs with the format expected by speech/multimodal models (reducing distribution mismatch with respect to pretraining), lowers computational and memory costs, and ensures experimental reproducibility.


In [ ]:
def preprocess_audio_file(input_file, output_file=None, target_sr=16000):
    if output_file is None:
        output_file = input_file
    audio, _ = librosa.load(
        input_file,
        sr=target_sr,
        mono=True
    )
    audio = audio.astype(np.float32)
    sf.write(output_file, audio, target_sr)

Test

In [ ]:
PREPROCESS_DIR = os.path.join(DRIVE_DIR, "data", "audio_preprocessed")
os.makedirs(PREPROCESS_DIR, exist_ok=True)


if DOWNLOAD:
    for item in tqdm(all_data, desc="Preprocessing all audio"):
        input_file = item['audio_path']
        output_file = os.path.join(PREPROCESS_DIR, f"{os.path.basename(input_file)}")
        preprocess_audio_file(input_file, output_file)
        item['audio_path'] = output_file
else:
    for item in all_data:
        input_file = os.path.join(PREPROCESS_DIR, f"{item['id']}.wav")
        if os.path.exists(input_file):
            item['audio_path'] = input_file
        else:
            raise FileNotFoundError(f"Preprocessed audio file for {item['id']} not found in {PREPROCESS_DIR}")

Train

In [ ]:
PREPROCESS_DIR = os.path.join(DRIVE_DIR, "data", "audio_train_preprocessed")
os.makedirs(PREPROCESS_DIR, exist_ok=True)


if DOWNLOAD:
    for item in tqdm(all_data_train, desc="Preprocessing all audio"):
        input_file = item['audio_path']
        output_file = os.path.join(PREPROCESS_DIR, f"{os.path.basename(input_file)}")
        preprocess_audio_file(input_file, output_file)
        item['audio_path'] = output_file
else:
    for item in all_data_train:
        input_file = os.path.join(PREPROCESS_DIR, f"{item['id']}.wav")
        if os.path.exists(input_file):
            item['audio_path'] = input_file
        else:
            raise FileNotFoundError(f"Preprocessed audio file for {item['id']} not found in {PREPROCESS_DIR}")

Preprocessing all audio: 100%|██████████| 900/900 [01:57<00:00,  7.67it/s]


### Preprocessing on text

Test

In [ ]:
ALL_DATA_PROCESSED_JSON = os.path.join(DATASET, "all_data_processed.json")

all_data_processed = []
for item in all_data:
    processed_item = item.copy()
    processed_item["text"] = item["text"][:500]
    all_data_processed.append(processed_item)

with open(ALL_DATA_PROCESSED_JSON, "w", encoding="utf-8") as f:
    json.dump(all_data_processed, f, ensure_ascii=False, indent=2)

print(f"Saved processed data to {ALL_DATA_PROCESSED_JSON}")

Saved processed data to /content/drive/MyDrive/TextMiningProject/data/all_data_processed.json


Train

In [ ]:
ALL_DATA_PROCESSED_JSON = os.path.join(DATASET, "all_data_train_processed.json")

all_data_train_processed = []
for item in all_data_train:
    processed_item = item.copy()
    processed_item["text"] = item["text"][:500]
    all_data_train_processed.append(processed_item)

with open(ALL_DATA_PROCESSED_JSON, "w", encoding="utf-8") as f:
    json.dump(all_data_train_processed, f, ensure_ascii=False, indent=2)

print(f"Saved processed data to {ALL_DATA_PROCESSED_JSON}")

Saved processed data to /content/drive/MyDrive/TextMiningProject/data/all_data_train_processed.json


# 3. Evaluation

**ROUGE** stands for Recall-Oriented Understudy for Gisting Evaluation.
ROUGE is the leading evaluation metric for text summarization that compares a model-generated summary against the reference human-annotated one.

In [6]:
!pip install -q rouge_score evaluate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [7]:
import evaluate, nltk

nltk.download('punkt')
nltk.download("punkt_tab")

rouge = evaluate.load("rouge")

def get_rouge_scores(predictions, references):

    result_rouge = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
    result = {k: round(v * 100, 2) for k, v in result_rouge.items()}
    return result

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


utilities

In [8]:
from collections import defaultdict
import re

def split_gold_by_dataset(all_data):
    gold_by_dataset = defaultdict(list)
    for item in all_data:
        gold_by_dataset[item["dataset"]].append(item["summary"])
    return gold_by_dataset


def load_predictions_with_dataset(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def split_predictions_by_dataset(data, summary_key):
    pred_by_dataset = {"xsum": [], "samsum": []}

    for item in data:
        pred_by_dataset[item["dataset"]].append(item[summary_key])

    return pred_by_dataset

def print_sep(n=100):
    print("-" * n)

def print_all_predictions_for_example(
    dataset_name,
    idx,
    gold_by_dataset,
    preds_by_dataset
):
    print_sep()
    print(f"DATASET: {dataset_name} | INDEX: {idx}")
    print_sep()

    print("GOLD STANDARD SUMMARY:\n")
    print(gold_by_dataset[dataset_name][idx])
    print_sep()

    for label, ds_dict in preds_by_dataset.items():
        print(f"{label.upper()}:\n")
        print(ds_dict[dataset_name][idx])
        print_sep()

gold standard

In [9]:
gold_by_dataset = split_gold_by_dataset(all_data)

## 3.1 Results of the Inference of Gemma and Qwen

### Load Results

Test

In [10]:
GEMMA_RESULTS_DIR = os.path.join(DRIVE_DIR, "gemma")
QWEN_RESULTS_DIR = os.path.join(DRIVE_DIR, "qwen")

#gemma
gemma_text_summaries_greedy = os.path.join(GEMMA_RESULTS_DIR, "text_summaries_greedy.json")
gemma_text_summaries_temperature = os.path.join(GEMMA_RESULTS_DIR, "text_summaries_temperature.json")
gemma_text_summaries_beam = os.path.join(GEMMA_RESULTS_DIR, "text_summaries_beam.json")
gemma_audio_summaries_greedy = os.path.join(GEMMA_RESULTS_DIR, "audio_summaries_greedy.json")
gemma_audio_summaries_temperature = os.path.join(GEMMA_RESULTS_DIR, "audio_summaries_temperature.json")
gemma_audio_summaries_beam = os.path.join(GEMMA_RESULTS_DIR, "audio_summaries_beam.json")

gemma_results = [
    gemma_text_summaries_greedy,
    gemma_text_summaries_temperature,
    gemma_text_summaries_beam,
    gemma_audio_summaries_greedy,
    gemma_audio_summaries_temperature,
    gemma_audio_summaries_beam
]

#qwen
qwen_text_summaries_greedy = os.path.join(QWEN_RESULTS_DIR, "text_summaries_greedy.json")
qwen_text_summaries_temperature = os.path.join(QWEN_RESULTS_DIR, "text_summaries_temperature.json")
qwen_text_summaries_beam = os.path.join(QWEN_RESULTS_DIR, "text_summaries_beam.json")
qwen_audio_summaries_greedy = os.path.join(QWEN_RESULTS_DIR, "audio_summaries_greedy.json")
qwen_audio_summaries_temperature = os.path.join(QWEN_RESULTS_DIR, "audio_summaries_temperature.json")
qwen_audio_summaries_beam = os.path.join(QWEN_RESULTS_DIR, "audio_summaries_beam.json")

qwen_results = [
    qwen_text_summaries_greedy,
    qwen_text_summaries_temperature,
    qwen_text_summaries_beam,
    qwen_audio_summaries_greedy,
    qwen_audio_summaries_temperature,
    qwen_audio_summaries_beam
]

### Cleaning the predictions


In [ ]:
# gemma
if CLEAN:
    for results in gemma_results:
        with open(results, "r", encoding="utf-8") as f:
            data = json.load(f)

        if results in [gemma_text_summaries_greedy, gemma_text_summaries_temperature, gemma_text_summaries_beam]:
            summary_key = "summary_gemma_text"
        else:
            summary_key = "summary_gemma_audio"

        for item in data:
            pred = item[summary_key]
            pred = re.sub(r"<.*?>", "", pred)

            pred = pred.replace("\n", " ").replace("\\", "")

            pred = re.sub(r"\s+", " ", pred)

            pred = pred.strip()
            item[summary_key] = pred

        with open(results, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    cleaned_dir = "/content/drive/MyDrive/TextMiningProject/qwen/cleaned_results"
    os.makedirs(cleaned_dir, exist_ok=True)

In [ ]:
#qwen
if CLEAN:
    for results in qwen_results:
        with open(results, "r", encoding="utf-8") as f:
            data = json.load(f)

        if results in [qwen_text_summaries_greedy, qwen_text_summaries_temperature, qwen_text_summaries_beam]:
            summary_key = "summary_qwen_text"
        else:
            summary_key = "summary_qwen_audio"

        for item in data:
            pred = item[summary_key]

            assistant_split = re.split(r"assistant\n", pred, flags=re.IGNORECASE)
            cleaned = assistant_split[1]
            new_line_split = re.split(r"\n", cleaned, flags=re.IGNORECASE)
            cleaned = new_line_split[0]
            item[summary_key] = cleaned

        base_name = os.path.basename(results)
        name, ext = os.path.splitext(base_name)
        new_file = os.path.join(cleaned_dir, f"{name}_cleaned{ext}")

        with open(new_file, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print(f"Saved cleaned results to {new_file}")

Saved cleaned results to /content/drive/MyDrive/TextMiningProject/qwen/cleaned_results/text_summaries_greedy_cleaned.json
Saved cleaned results to /content/drive/MyDrive/TextMiningProject/qwen/cleaned_results/text_summaries_temperature_cleaned.json
Saved cleaned results to /content/drive/MyDrive/TextMiningProject/qwen/cleaned_results/text_summaries_beam_cleaned.json
Saved cleaned results to /content/drive/MyDrive/TextMiningProject/qwen/cleaned_results/audio_summaries_greedy_cleaned.json
Saved cleaned results to /content/drive/MyDrive/TextMiningProject/qwen/cleaned_results/audio_summaries_temperature_cleaned.json
Saved cleaned results to /content/drive/MyDrive/TextMiningProject/qwen/cleaned_results/audio_summaries_beam_cleaned.json


### Gemma

In [14]:
gemma_configs = {
    ("text", "greedy"): gemma_text_summaries_greedy,
    ("text", "temperature"): gemma_text_summaries_temperature,
    ("text", "beam"): gemma_text_summaries_beam,
    ("audio", "greedy"): gemma_audio_summaries_greedy,
    ("audio", "temperature"): gemma_audio_summaries_temperature,
    ("audio", "beam"): gemma_audio_summaries_beam,
}

rows = []

for (modality, decoding), path in gemma_configs.items():
    summary_key = f"summary_gemma_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    rouge_xsum = get_rouge_scores(
        predictions=pred_by_dataset["xsum"],
        references=gold_by_dataset["xsum"]
    )

    rouge_samsum = get_rouge_scores(
        predictions=pred_by_dataset["samsum"],
        references=gold_by_dataset["samsum"]
    )

    rows.append({
        "model": "gemma",
        "modality": modality,
        "decoding": decoding,
        "xsum_rouge1": rouge_xsum["rouge1"],
        "xsum_rouge2": rouge_xsum["rouge2"],
        "xsum_rougeL": rouge_xsum["rougeL"],
        "xsum_rougeLsum": rouge_xsum["rougeLsum"],
        "samsum_rouge1": rouge_samsum["rouge1"],
        "samsum_rouge2": rouge_samsum["rouge2"],
        "samsum_rougeL": rouge_samsum["rougeL"],
        "samsum_rougeLsum": rouge_samsum["rougeLsum"],
    })

gemma_rouge_table = pd.DataFrame(rows)
gemma_rouge_table



,model,modality,decoding,xsum_rouge1,xsum_rouge2,xsum_rougeL,xsum_rougeLsum,samsum_rouge1,samsum_rouge2,samsum_rougeL,samsum_rougeLsum
0,gemma,text,greedy,22.83,3.94,16.21,16.16,40.63,15.34,32.12,32.06
1,gemma,text,temperature,23.00,3.97,16.33,16.32,39.89,14.93,31.91,31.85
2,gemma,text,beam,22.74,3.72,16.17,16.14,41.07,15.46,32.83,32.83
3,gemma,audio,greedy,20.59,3.25,14.62,14.61,33.91,10.41,26.09,26.12
4,gemma,audio,temperature,21.17,3.13,14.87,14.85,33.35,10.19,25.77,25.76
5,gemma,audio,beam,20.88,3.14,14.57,14.55,34.82,10.97,26.73,26.77


Let's print a predicted summary along its human-annotated one.

In [ ]:
preds_gemma = {}

for (modality, decoding), path in gemma_configs.items():
    summary_key = f"summary_gemma_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    label = f"gemma_{modality}_{decoding}"
    preds_gemma[label] = {
        "xsum": pred_by_dataset.get("xsum", []),
        "samsum": pred_by_dataset.get("samsum", [])
    }


In [ ]:
print_all_predictions_for_example(
    dataset_name="xsum",
    idx=0,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_gemma
)

----------------------------------------------------------------------------------------------------
DATASET: xsum | INDEX: 0
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_GREEDY:

In April 2013, a minibus accident in Liverpool killed 18-year-old Bethany Jones and seriously injured Sarah Johnson and others, prompting Johnson to seek support from a charity and ultimately inspire her to advocate for others.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_TEMPERATURE:

In April 2013, a minibus accident in Liverpool killed 18-year-old Bethany Jones, and Sarah Johnson, who was among the injured, later sought supp

In [ ]:
print_all_predictions_for_example(
    dataset_name="samsum",
    idx=0,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_gemma
)

----------------------------------------------------------------------------------------------------
DATASET: samsum | INDEX: 0
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

Both Claire and Linda are making curry for dinner. 
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_GREEDY:

The friends Claire, Kim, and Linda are sharing photos of their meals, with Kim and Claire expressing enjoyment of Linda's curry.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_TEMPERATURE:

The text shows a playful exchange between Claire, Kim, and Linda about their respective dinners, with Claire and Kim expressing admiration for Linda's cooking.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_BEAM:

The friends Claire, Kim, and Linda are s

### Observations

1. **Text vs Audio Modality**

   * Across both datasets, **text inputs consistently outperform audio inputs**.
   * For XSum, ROUGE-1 drops from ~23 for text to ~21 for audio.
   * For SAMSum, the drop is even larger: ROUGE-1 from ~41 (text) to ~34 (audio).
   * This suggests the model is better at capturing document-level content from **clean textual input**, while audio processing introduces noise or loses information.

2. **Decoding Strategy Impact**

   * Differences between **greedy, temperature, and beam search** are small for text, especially on XSum.
   * Beam search slightly improves SAMSum ROUGE scores for text (ROUGE-1: 41.10 vs 40.60 for greedy).
   * For audio, **temperature sampling** sometimes gives slightly better ROUGE-1 than greedy, but overall differences are modest.

3. **Dataset-Specific Trends**

   * XSum (news articles) shows lower ROUGE scores overall (~16 ROUGE-L) compared to SAMSum (dialogues, ~32 ROUGE-L), reflecting **different summary lengths and complexity**.
   * Audio modality suffers more on SAMSum than XSum, indicating that **multi-speaker dialogue audio is harder to summarize accurately**.

4. **ROUGE-LSUM vs ROUGE-L**

   * ROUGE-LSUM and ROUGE-L are almost identical, showing that **longest common subsequence metrics over multiple sentences are consistent with standard L** in this setting.



### Qwen

In [12]:
QWEN_RESULTS_DIR_CLEANED = os.path.join(DRIVE_DIR, "qwen", "cleaned_results")

qwen_text_summaries_greedy = os.path.join(QWEN_RESULTS_DIR_CLEANED, "text_summaries_greedy_cleaned.json")
gwen_text_summaries_temperature = os.path.join(QWEN_RESULTS_DIR_CLEANED, "text_summaries_temperature_cleaned.json")
qwen_text_summaries_beam = os.path.join(QWEN_RESULTS_DIR_CLEANED, "text_summaries_beam_cleaned.json")
qwen_audio_summaries_greedy = os.path.join(QWEN_RESULTS_DIR_CLEANED, "audio_summaries_greedy_cleaned.json")
qwen_audio_summaries_temperature = os.path.join(QWEN_RESULTS_DIR_CLEANED, "audio_summaries_temperature_cleaned.json")
qwen_audio_summaries_beam = os.path.join(QWEN_RESULTS_DIR_CLEANED, "audio_summaries_beam_cleaned.json")

qwen_results = [
    qwen_text_summaries_greedy,
    qwen_text_summaries_temperature,
    qwen_text_summaries_beam,
    qwen_audio_summaries_greedy,
    qwen_audio_summaries_temperature,
    qwen_audio_summaries_beam
]

In [15]:
gwen_configs = {
    ("text", "greedy"): qwen_text_summaries_greedy,
    ("text", "temperature"): gwen_text_summaries_temperature,
    ("text", "beam"): qwen_text_summaries_beam,
    ("audio", "greedy"): qwen_audio_summaries_greedy,
    ("audio", "temperature"): qwen_audio_summaries_temperature,
    ("audio", "beam"): qwen_audio_summaries_beam,
}

rows = []

for (modality, decoding), path in gwen_configs.items():
    summary_key = f"summary_qwen_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    rouge_xsum = get_rouge_scores(
        predictions=pred_by_dataset["xsum"],
        references=gold_by_dataset["xsum"]
    )

    rouge_samsum = get_rouge_scores(
        predictions=pred_by_dataset["samsum"],
        references=gold_by_dataset["samsum"]
    )

    rows.append({
        "model": "qwen",
        "modality": modality,
        "decoding": decoding,
        "xsum_rouge1": rouge_xsum["rouge1"],
        "xsum_rouge2": rouge_xsum["rouge2"],
        "xsum_rougeL": rouge_xsum["rougeL"],
        "xsum_rougeLsum": rouge_xsum["rougeLsum"],
        "samsum_rouge1": rouge_samsum["rouge1"],
        "samsum_rouge2": rouge_samsum["rouge2"],
        "samsum_rougeL": rouge_samsum["rougeL"],
        "samsum_rougeLsum": rouge_samsum["rougeLsum"],
    })

qwen_rouge_table = pd.DataFrame(rows)
qwen_rouge_table



,model,modality,decoding,xsum_rouge1,xsum_rouge2,xsum_rougeL,xsum_rougeLsum,samsum_rouge1,samsum_rouge2,samsum_rougeL,samsum_rougeLsum
0,qwen,text,greedy,23.70,5.19,17.83,17.81,41.79,16.10,33.48,33.51
1,qwen,text,temperature,22.46,4.00,16.56,16.51,36.16,11.58,28.33,28.41
2,qwen,text,beam,22.44,4.39,16.61,16.59,41.98,16.46,33.58,33.55
3,qwen,audio,greedy,19.50,2.61,14.66,14.69,34.47,11.33,27.46,27.45
4,qwen,audio,temperature,20.04,3.22,14.89,14.92,31.45,9.18,24.72,24.68
5,qwen,audio,beam,20.64,3.66,15.43,15.46,36.85,13.08,29.10,29.08


In [ ]:
preds_qwen = {}

for (modality, decoding), path in gwen_configs.items():
    summary_key = f"summary_qwen_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    label = f"qwen_{modality}_{decoding}"
    preds_qwen[label] = {
        "xsum": pred_by_dataset.get("xsum", []),
        "samsum": pred_by_dataset.get("samsum", [])
    }


In [ ]:
print_all_predictions_for_example(
    dataset_name="xsum",
    idx=0,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_qwen
)

----------------------------------------------------------------------------------------------------
DATASET: xsum | INDEX: 0
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.
----------------------------------------------------------------------------------------------------
QWEN_TEXT_GREEDY:

Sarah Johnson, a survivor of a tragic bus crash, now wants to help others after receiving support from a charity while in hospital.
----------------------------------------------------------------------------------------------------
QWEN_TEXT_TEMPERATURE:

Sarah Johnson was a bus driver killed in a fatal M62 crash in 2013, which led her to establish a charity that supports others following similar incidents.
---------------------------------------------------------------------------------------

In [ ]:
print_all_predictions_for_example(
    dataset_name="samsum",
    idx=0,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_qwen
)

----------------------------------------------------------------------------------------------------
DATASET: samsum | INDEX: 0
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

Both Claire and Linda are making curry for dinner. 
----------------------------------------------------------------------------------------------------
QWEN_TEXT_GREEDY:

Claire, Kim, and Linda are having a fun dinner party with Linda's curry recipe.
----------------------------------------------------------------------------------------------------
QWEN_TEXT_TEMPERATURE:

Claire is surprised by the beautiful dinner prepared by Linda, who is proud of her curry creation.
----------------------------------------------------------------------------------------------------
QWEN_TEXT_BEAM:

Claire, Kim, and Linda are comparing their cooking skills, with Linda boasting about her curry dish.
----------------------------------------------------



### **Observations for QWEN**

1. **Text vs Audio Modality**

   * Text inputs consistently outperform audio inputs across both datasets.
   * For XSum, ROUGE-1 drops from ~23.7 for text (`greedy`) to ~19.5 for audio (`greedy`).
   * For SAMSum, the drop is more pronounced: ROUGE-1 from ~41.8 (text) to ~34.5 (audio).
   * This indicates that QWEN handles clean textual input better, while audio introduces noise or information loss.

2. **Decoding Strategy Impact**

   * Differences between **greedy, temperature, and beam search** are more variable than GEMMA.
   * For text:

     * Greedy decoding gives the highest XSum ROUGE-1 (23.70) and high SAMSum scores.
     * Temperature sampling sometimes reduces ROUGE scores significantly (e.g., text SAMSum ROUGE-1 drops to 36.16).
     * Beam search is comparable to greedy for text and can slightly improve ROUGE-L metrics.
   * For audio:

     * Beam decoding provides the highest ROUGE scores among audio inputs (e.g., SAMSum ROUGE-Lsum = 29.08), outperforming greedy and temperature.
     * Temperature sampling introduces variability but can occasionally improve XSum ROUGE-1 slightly compared to greedy (20.04 vs 19.50).

#### Qwen VS Gemma

In [16]:
combined_gemma_qwen_rouge_table = pd.concat([
    gemma_rouge_table,
    qwen_rouge_table
], ignore_index=True)

print("Combined Gemma and Qwen ROUGE score")
display(combined_gemma_qwen_rouge_table)

Combined Gemma and Qwen ROUGE score


,model,modality,decoding,xsum_rouge1,xsum_rouge2,xsum_rougeL,xsum_rougeLsum,samsum_rouge1,samsum_rouge2,samsum_rougeL,samsum_rougeLsum
0,gemma,text,greedy,22.83,3.94,16.21,16.16,40.63,15.34,32.12,32.06
1,gemma,text,temperature,23.00,3.97,16.33,16.32,39.89,14.93,31.91,31.85
2,gemma,text,beam,22.74,3.72,16.17,16.14,41.07,15.46,32.83,32.83
3,gemma,audio,greedy,20.59,3.25,14.62,14.61,33.91,10.41,26.09,26.12
4,gemma,audio,temperature,21.17,3.13,14.87,14.85,33.35,10.19,25.77,25.76
5,gemma,audio,beam,20.88,3.14,14.57,14.55,34.82,10.97,26.73,26.77
6,qwen,text,greedy,23.70,5.19,17.83,17.81,41.79,16.10,33.48,33.51
7,qwen,text,temperature,22.46,4.00,16.56,16.51,36.16,11.58,28.33,28.41
8,qwen,text,beam,22.44,4.39,16.61,16.59,41.98,16.46,33.58,33.55
9,qwen,audio,greedy,19.50,2.61,14.66,14.69,34.47,11.33,27.46,27.45




## **Aggregated Observations: GEMMA vs QWEN**

### 1. **Text Modality Trends**

* **QWEN generally outperforms GEMMA** on text summaries across both datasets, especially for `xsum_rouge1` and `xsum_rougeLsum`.

  * Example: XSum greedy `xsum_rouge1`: 23.70 (QWEN) vs 22.83 (GEMMA)
  * SAMSum greedy `samsum_rougeLsum`: 33.51 (QWEN) vs 32.06 (GEMMA)

* **Beam search** improves some metrics slightly for both models:

  * GEMMA: SAMSum ROUGE-Lsum increases from 32.06 (greedy) → 32.83 (beam)
  * QWEN: XSum ROUGE-L increases marginally 17.81 → 16.59 (small drop in ROUGE1 vs L, shows decoding effect)

* **Temperature decoding** shows variability:

  * GEMMA: small differences across XSum/SAMSum metrics
  * QWEN: noticeable drops for SAMSum (ROUGE-1: 41.79 → 36.16), indicating sensitivity to stochastic decoding in dialogue data.

---

### 2. **Audio Modality Trends**

* **Performance drops for audio** in both models, reflecting the added difficulty of summarizing audio inputs.

  * Example XSum greedy: 20.59 (GEMMA) vs 19.50 (QWEN)
  * Example SAMSum greedy: 26.12 (GEMMA) vs 27.45 (QWEN, slight edge in beam decoding)

* **Beam decoding** generally recovers performance in audio summaries:

  * GEMMA audio beam `samsum_rougeLsum`: 26.77
  * QWEN audio beam `samsum_rougeLsum`: 29.08

* Temperature sampling introduces more variance, especially in QWEN audio SAMSum (ROUGE-1 drops to 31.45).

---

### 3. **Dataset-Specific Observations**

* **XSum (news)**

  * Shows smaller differences between text and audio than SAMSum.
  * Both models achieve moderate ROUGE-L scores (~16–17), with QWEN slightly ahead in text.

* **SAMSum (dialogues)**

  * Larger gap between text and audio, reflecting the challenge of multi-speaker audio summaries.
  * QWEN text performs very well (ROUGE-1 ~41–42), but audio suffers unless beam decoding is used.
  * GEMMA is more consistent but slightly lower overall.





## 3.2 Results of Gemma Fine-Tuning

Attention!! When the sampling rate is not specified is assumed 16000kHz

clean and load the results

In [17]:
GEMMA_RESULTS_DIR_FINE_TUNING_TEXT = os.path.join(DRIVE_DIR, "gemma_fine_tuning_text")
GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO = os.path.join(DRIVE_DIR, "gemma_fine_tuning_audio")
GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24 = os.path.join(DRIVE_DIR, "gemma_fine_tuning_audio_sr24")
GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8 = os.path.join(DRIVE_DIR, "gemma_fine_tuning_audio_sr8")

# gemma fine-tuning text
gemma_text_summaries_greedy_fine_text = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_TEXT, "text_summaries_greedy.json")
gemma_text_summaries_temperature_fine_text = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_TEXT, "text_summaries_temperature.json")
gemma_text_summaries_beam_fine_text = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_TEXT, "text_summaries_beam.json")
gemma_audio_summaries_greedy_fine_text = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_TEXT, "audio_summaries_greedy.json")
gemma_audio_summaries_temperature_fine_text = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_TEXT, "audio_summaries_temperature.json")
gemma_audio_summaries_beam_fine_text = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_TEXT, "audio_summaries_beam.json")

gemma_fine_text = [
    gemma_text_summaries_greedy_fine_text,
    gemma_text_summaries_temperature_fine_text,
    gemma_text_summaries_beam_fine_text,
    gemma_audio_summaries_greedy_fine_text,
    gemma_audio_summaries_temperature_fine_text,
    gemma_audio_summaries_beam_fine_text
]

# gemma fine-tuning audio sr=16000
gemma_text_summaries_greedy_fine_audio = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO, "text_summaries_greedy.json")
gemma_text_summaries_temperature_fine_audio = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO, "text_summaries_temperature.json")
gemma_text_summaries_beam_fine_audio = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO, "text_summaries_beam.json")
gemma_audio_summaries_greedy_fine_audio = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO, "audio_summaries_greedy.json")
gemma_audio_summaries_temperature_fine_audio = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO, "audio_summaries_temperature.json")
gemma_audio_summaries_beam_fine_audio = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO, "audio_summaries_beam.json")

gemma_fine_audio = [
    gemma_text_summaries_greedy_fine_audio,
    gemma_text_summaries_temperature_fine_audio,
    gemma_text_summaries_beam_fine_audio,
    gemma_audio_summaries_greedy_fine_audio,
    gemma_audio_summaries_temperature_fine_audio,
    gemma_audio_summaries_beam_fine_audio
]

# gemma fine-tuning audio sr=24000
gemma_text_summaries_greedy_fine_audio_sr24 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24, "text_summaries_greedy.json")
gemma_text_summaries_temperature_fine_audio_sr24 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24, "text_summaries_temperature.json")
gemma_text_summaries_beam_fine_audio_sr24 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24, "text_summaries_beam.json")
gemma_audio_summaries_greedy_fine_audio_sr24 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24, "audio_summaries_greedy.json")
gemma_audio_summaries_temperature_fine_audio_sr24 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24, "audio_summaries_temperature.json")
gemma_audio_summaries_beam_fine_audio_sr24 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24, "audio_summaries_beam.json")

gemma_fine_audio_sr24 = [
    gemma_text_summaries_greedy_fine_audio_sr24,
    gemma_text_summaries_temperature_fine_audio_sr24,
    gemma_text_summaries_beam_fine_audio_sr24,
    gemma_audio_summaries_greedy_fine_audio_sr24,
    gemma_audio_summaries_temperature_fine_audio_sr24,
    gemma_audio_summaries_beam_fine_audio_sr24
]

# gemma fine-tuning audio sr=8000
gemma_text_summaries_greedy_fine_audio_sr8 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8, "text_summaries_greedy.json")
gemma_text_summaries_temperature_fine_audio_sr8 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8, "text_summaries_temperature.json")
gemma_text_summaries_beam_fine_audio_sr8 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8, "text_summaries_beam.json")
gemma_audio_summaries_greedy_fine_audio_sr8 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8, "audio_summaries_greedy.json")
gemma_audio_summaries_temperature_fine_audio_sr8 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8, "audio_summaries_temperature.json")
gemma_audio_summaries_beam_fine_audio_sr8 = os.path.join(GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8, "audio_summaries_beam.json")

gemma_fine_audio_sr8 = [
    gemma_text_summaries_greedy_fine_audio_sr8,
    gemma_text_summaries_temperature_fine_audio_sr8,
    gemma_text_summaries_beam_fine_audio_sr8,
    gemma_audio_summaries_greedy_fine_audio_sr8,
    gemma_audio_summaries_temperature_fine_audio_sr8,
    gemma_audio_summaries_beam_fine_audio_sr8
]

In [ ]:
# gemma fine-tuning text
if CLEAN:
    for results in gemma_fine_text:
        with open(results, "r", encoding="utf-8") as f:
            data = json.load(f)

        if results in [gemma_text_summaries_greedy_fine_text, gemma_text_summaries_temperature_fine_text, gemma_text_summaries_beam_fine_text]:
            summary_key = "summary_gemma_text"
        else:
            summary_key = "summary_gemma_audio"

        for item in data:
            pred = item[summary_key]
            pred = re.sub(r"<.*?>", "", pred)

            pred = pred.replace("\n", " ").replace("\\", "")

            pred = re.sub(r"\s+", " ", pred)

            pred = pred.strip()
            item[summary_key] = pred

        with open(results, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
# gemma fine-tuning audio sr=16000
if CLEAN:
    for results in gemma_fine_audio:
        with open(results, "r", encoding="utf-8") as f:
            data = json.load(f)

        if results in [gemma_text_summaries_greedy_fine_audio, gemma_text_summaries_temperature_fine_audio, gemma_text_summaries_beam_fine_audio]:
            summary_key = "summary_gemma_text"
        else:
            summary_key = "summary_gemma_audio"

        for item in data:
            pred = item[summary_key]
            pred = re.sub(r"<.*?>", "", pred)

            pred = pred.replace("\n", " ").replace("\\", "")

            pred = re.sub(r"\s+", " ", pred)

            pred = pred.strip()
            item[summary_key] = pred

        with open(results, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
# gemma fine-tuning audio sr=24000
if CLEAN:
    for results in gemma_fine_audio_sr24:
        with open(results, "r", encoding="utf-8") as f:
            data = json.load(f)

        if results in [gemma_text_summaries_greedy_fine_audio_sr24, gemma_text_summaries_temperature_fine_audio_sr24, gemma_text_summaries_beam_fine_audio_sr24]:
            summary_key = "summary_gemma_text"
        else:
            summary_key = "summary_gemma_audio"

        for item in data:
            pred = item[summary_key]
            pred = re.sub(r"<.*?>", "", pred)

            pred = pred.replace("\n", " ").replace("\\", "")

            pred = re.sub(r"\s+", " ", pred)

            pred = pred.strip()
            item[summary_key] = pred

        with open(results, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
# gemma fine-tuning audio sr=8000
if CLEAN:
    for results in gemma_fine_audio_sr8:
        with open(results, "r", encoding="utf-8") as f:
            data = json.load(f)

        if results in [gemma_text_summaries_greedy_fine_audio_sr8, gemma_text_summaries_temperature_fine_audio_sr8, gemma_text_summaries_beam_fine_audio_sr8]:
            summary_key = "summary_gemma_text"
        else:
            summary_key = "summary_gemma_audio"

        for item in data:
            pred = item[summary_key]
            pred = re.sub(r"<.*?>", "", pred)

            pred = pred.replace("\n", " ").replace("\\", "")

            pred = re.sub(r"\s+", " ", pred)

            pred = pred.strip()
            item[summary_key] = pred

        with open(results, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

### Gemma text fine tuning

In [18]:
gemma_text_configs = {
    ("text", "greedy"): gemma_text_summaries_greedy_fine_text,
    ("text", "temperature"): gemma_text_summaries_temperature_fine_text,
    ("text", "beam"): gemma_text_summaries_beam_fine_text,
    ("audio", "greedy"): gemma_audio_summaries_greedy_fine_text,
    ("audio", "temperature"): gemma_audio_summaries_temperature_fine_text,
    ("audio", "beam"): gemma_audio_summaries_beam_fine_text,
}

rows = []

for (modality, decoding), path in gemma_text_configs.items():
    summary_key = f"summary_gemma_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    rouge_xsum = get_rouge_scores(
        predictions=pred_by_dataset["xsum"],
        references=gold_by_dataset["xsum"]
    )

    rouge_samsum = get_rouge_scores(
        predictions=pred_by_dataset["samsum"],
        references=gold_by_dataset["samsum"]
    )

    rows.append({
        "model": "gemma_text_fn",
        "modality": modality,
        "decoding": decoding,
        "xsum_rouge1": rouge_xsum["rouge1"],
        "xsum_rouge2": rouge_xsum["rouge2"],
        "xsum_rougeL": rouge_xsum["rougeL"],
        "xsum_rougeLsum": rouge_xsum["rougeLsum"],
        "samsum_rouge1": rouge_samsum["rouge1"],
        "samsum_rouge2": rouge_samsum["rouge2"],
        "samsum_rougeL": rouge_samsum["rougeL"],
        "samsum_rougeLsum": rouge_samsum["rougeLsum"],
    })

gemma_rouge_table_text_fn = pd.DataFrame(rows)
gemma_rouge_table_text_fn



,model,modality,decoding,xsum_rouge1,xsum_rouge2,xsum_rougeL,xsum_rougeLsum,samsum_rouge1,samsum_rouge2,samsum_rougeL,samsum_rougeLsum
0,gemma_text_fn,text,greedy,15.86,4.25,12.71,12.68,39.42,17.79,32.10,32.05
1,gemma_text_fn,text,temperature,17.84,4.46,13.56,13.51,39.52,18.35,32.54,32.56
2,gemma_text_fn,text,beam,24.06,6.13,18.75,18.74,43.43,20.55,35.61,35.64
3,gemma_text_fn,audio,greedy,13.88,0.89,11.24,11.20,8.32,0.50,7.01,7.04
4,gemma_text_fn,audio,temperature,12.59,0.71,10.29,10.29,8.53,0.36,7.06,7.09
5,gemma_text_fn,audio,beam,13.88,0.89,11.24,11.20,8.32,0.50,7.01,7.04


In [ ]:
preds_gemma_text_fn = {}

for (modality, decoding), path in gemma_text_configs.items():
    summary_key = f"summary_gemma_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    label = f"gemma_{modality}_{decoding}"
    preds_gemma_text_fn[label] = {
        "xsum": pred_by_dataset.get("xsum", []),
        "samsum": pred_by_dataset.get("samsum", [])
    }


In [ ]:
print_all_predictions_for_example(
    dataset_name="xsum",
    idx=0,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_gemma_text_fn
)

----------------------------------------------------------------------------------------------------
DATASET: xsum | INDEX: 0
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

A woman who was seriously hurt in a fatal hen party motorway crash is now helping other major trauma victims rebuild their lives.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_GREEDY:

>Sarah Johnson is raising money to support other women who have been injured in road accidents.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_TEMPERATURE:

>Sarah Johnson is raising money to support other women who have been hurt in the minibus accident.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_BEAM:

>Sarah Johnson is supporting a charity that helped her aft

In [ ]:
print_all_predictions_for_example(
    dataset_name="samsum",
    idx=1,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_gemma_text_fn
)

----------------------------------------------------------------------------------------------------
DATASET: samsum | INDEX: 1
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

Derek and Alyssa make fun of Fergie's performance of the national anthem.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_GREEDY:

Derek and Alyssa are laughing at Fergie's national anthem performance.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_TEMPERATURE:

Derek and Alyssa are laughing at Fergie's national anthem performance.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_BEAM:

Derek and Alyssa are laughing at Fergie's National Anthem.
----------------------------------------------------------------------------------------------------
GEMMA

### Gemma audio fine tuning sr=16000

In [19]:
gemma_audio_configs = {
    ("text", "greedy"): gemma_text_summaries_greedy_fine_audio,
    ("text", "temperature"): gemma_text_summaries_temperature_fine_audio,
    ("text", "beam"): gemma_text_summaries_beam_fine_audio,
    ("audio", "greedy"): gemma_audio_summaries_greedy_fine_audio,
    ("audio", "temperature"): gemma_audio_summaries_temperature_fine_audio,
    ("audio", "beam"): gemma_audio_summaries_beam_fine_audio,
}

rows = []

for (modality, decoding), path in gemma_audio_configs.items():
    summary_key = f"summary_gemma_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    rouge_xsum = get_rouge_scores(
        predictions=pred_by_dataset["xsum"],
        references=gold_by_dataset["xsum"]
    )

    rouge_samsum = get_rouge_scores(
        predictions=pred_by_dataset["samsum"],
        references=gold_by_dataset["samsum"]
    )

    rows.append({
        "model": "gemma_audio_fn",
        "modality": modality,
        "decoding": decoding,
        "xsum_rouge1": rouge_xsum["rouge1"],
        "xsum_rouge2": rouge_xsum["rouge2"],
        "xsum_rougeL": rouge_xsum["rougeL"],
        "xsum_rougeLsum": rouge_xsum["rougeLsum"],
        "samsum_rouge1": rouge_samsum["rouge1"],
        "samsum_rouge2": rouge_samsum["rouge2"],
        "samsum_rougeL": rouge_samsum["rougeL"],
        "samsum_rougeLsum": rouge_samsum["rougeLsum"],
    })

gemma_rouge_table_audio_fn = pd.DataFrame(rows)
gemma_rouge_table_audio_fn



,model,modality,decoding,xsum_rouge1,xsum_rouge2,xsum_rougeL,xsum_rougeLsum,samsum_rouge1,samsum_rouge2,samsum_rougeL,samsum_rougeLsum
0,gemma_audio_fn,text,greedy,14.87,3.47,11.63,11.64,16.58,6.67,13.41,13.46
1,gemma_audio_fn,text,temperature,16.27,3.77,12.44,12.40,18.35,7.50,14.95,15.16
2,gemma_audio_fn,text,beam,21.47,4.73,16.51,16.54,29.75,12.27,23.96,24.03
3,gemma_audio_fn,audio,greedy,12.13,0.69,8.87,8.84,11.30,0.41,8.08,8.10
4,gemma_audio_fn,audio,temperature,13.48,0.94,10.68,10.67,9.67,0.36,7.69,7.69
5,gemma_audio_fn,audio,beam,12.13,0.69,8.87,8.84,11.30,0.41,8.08,8.10


In [ ]:
preds_gemma_audio_fn = {}

for (modality, decoding), path in gemma_audio_configs.items():
    summary_key = f"summary_gemma_{modality}"

    data = load_predictions_with_dataset(path)
    pred_by_dataset = split_predictions_by_dataset(data, summary_key)

    label = f"gemma_{modality}_{decoding}"
    preds_gemma_audio_fn[label] = {
        "xsum": pred_by_dataset.get("xsum", []),
        "samsum": pred_by_dataset.get("samsum", [])
    }


In [ ]:
print_all_predictions_for_example(
    dataset_name="xsum",
    idx=2,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_gemma_audio_fn
)

----------------------------------------------------------------------------------------------------
DATASET: xsum | INDEX: 2
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

Walt Disney World has unveiled a lighthouse memorial for a young boy who was killed by an alligator while on holiday at the Florida theme park.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_GREEDY:

Alligators have been blamed for the death of a two-year-old boy in the Florida resort of Grand Floridian.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_TEMPERATURE:

Alligators are responsible for the most human fatalities in the United States, according to the U.S. Fish and Wildlife Service.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_BEAM:

Allig

In [ ]:
print_all_predictions_for_example(
    dataset_name="samsum",
    idx=0,
    gold_by_dataset=gold_by_dataset,
    preds_by_dataset=preds_gemma_audio_fn
)

----------------------------------------------------------------------------------------------------
DATASET: samsum | INDEX: 0
----------------------------------------------------------------------------------------------------
GOLD STANDARD SUMMARY:

Both Claire and Linda are making curry for dinner. 
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_GREEDY:

Linda is cooking curry. Claire and Kim are enjoying their dinner.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_TEMPERATURE:

The friends are cooking together. Claire is looking at Linda's dish.
----------------------------------------------------------------------------------------------------
GEMMA_TEXT_BEAM:

Linda is cooking curry. Claire and Kim are enjoying their dinner.
----------------------------------------------------------------------------------------------------
GEMMA_AUDIO_GREEDY:

A new 

## Final comment on the fine-tuning results

The results in the table illustrate a very clear (and unfortunately common) phenomenon in the world of multimodal models: Modality Collapse or Catastrophic Forgetting.

In [20]:
combined_gemma_rouge_table = pd.concat([
    gemma_rouge_table,
    gemma_rouge_table_text_fn,
    gemma_rouge_table_audio_fn
], ignore_index=True)

print("Combined Gemma ROUGE score table created.")
display(combined_gemma_rouge_table)

Combined Gemma ROUGE score table created.


,model,modality,decoding,xsum_rouge1,xsum_rouge2,xsum_rougeL,xsum_rougeLsum,samsum_rouge1,samsum_rouge2,samsum_rougeL,samsum_rougeLsum
0,gemma,text,greedy,22.83,3.94,16.21,16.16,40.63,15.34,32.12,32.06
1,gemma,text,temperature,23.00,3.97,16.33,16.32,39.89,14.93,31.91,31.85
2,gemma,text,beam,22.74,3.72,16.17,16.14,41.07,15.46,32.83,32.83
3,gemma,audio,greedy,20.59,3.25,14.62,14.61,33.91,10.41,26.09,26.12
4,gemma,audio,temperature,21.17,3.13,14.87,14.85,33.35,10.19,25.77,25.76
5,gemma,audio,beam,20.88,3.14,14.57,14.55,34.82,10.97,26.73,26.77
6,gemma_text_fn,text,greedy,15.86,4.25,12.71,12.68,39.42,17.79,32.10,32.05
7,gemma_text_fn,text,temperature,17.84,4.46,13.56,13.51,39.52,18.35,32.54,32.56
8,gemma_text_fn,text,beam,24.06,6.13,18.75,18.74,43.43,20.55,35.61,35.64
9,gemma_text_fn,audio,greedy,13.88,0.89,11.24,11.20,8.32,0.50,7.01,7.04


### Summary of Gemma Model Performance Analysis

1.  **Gemma Base vs. Fine-Tuned Models:**
    *   **Gemma Fine-tuned on Text (`gemma_text_fn`)**: This model shows mixed results. For text modality, the beam search decoding consistently achieves the highest ROUGE scores for both XSum (e.g., ROUGE-1: 24.06 vs 22.74 for base) and SAMSum (e.g., ROUGE-1: 43.35 vs 41.07 for base), significantly outperforming the base model in these specific configurations. However, when processing audio input, `gemma_text_fn` performs very poorly, often resulting in much lower ROUGE scores compared to the base Gemma model's audio performance, and in some cases, producing irrelevant summaries.
    *   **Gemma Fine-tuned on Audio (`gemma_audio_fn`)**: This model generally performs worse than the base Gemma model and `gemma_text_fn` across most configurations. For audio modality, its ROUGE scores are notably low. Even for text input, it doesn't surpass the base model's performance and is often worse than `gemma_text_fn`.

2.  **Impact of Modality (Text vs. Audio):**
    *   **Text inputs consistently outperform audio inputs** across all Gemma model variants (base, text_fn, audio_fn) and decoding strategies. This suggests that the models struggle more with processing information from audio, likely due to potential information loss or noise introduced during the audio-to-text conversion or inherent limitations in capturing nuanced information from speech.
    *   The drop in performance from text to audio is particularly severe for `gemma_text_fn`, indicating that fine-tuning solely on text data makes the model highly specialized and less robust to other modalities.

3.  **Effectiveness of Decoding Strategies:**
    *   **Beam Search** often yields the best performance among the decoding strategies, particularly for `gemma_text_fn` on text inputs. This is evident in the significant boosts in ROUGE scores for `gemma_text_fn` with beam search.
    *   **Greedy and Temperature sampling** perform similarly for the base Gemma model on text, with minor variations. However, their performance tends to be less consistent or lower than beam search when fine-tuning is applied, especially for `gemma_text_fn`.
    *   For audio inputs, the decoding strategy differences are less pronounced, likely overshadowed by the overall lower performance due to modality.

4.  **Dataset-Specific Trends (XSum vs. SAMSum):**
    *   **SAMSum** generally exhibits higher ROUGE scores than **XSum** across all models and configurations. This is expected, as SAMSum contains conversational dialogues, which tend to have more straightforward and shorter summaries, making them easier to generate accurately. XSum, with its news articles, demands more abstractive summarization, which is inherently more challenging.
    *   The performance degradation from text to audio input is relatively more significant for SAMSum, especially for the base Gemma model (ROUGE-1 from ~41 to ~34). This could imply that handling multi-speaker dialogues in audio format presents unique challenges for summarization, such as speaker diarization and turn-taking understanding, which might be harder to capture from audio alone.

In conclusion, fine-tuning Gemma specifically on text data with beam search can significantly improve text-based summarization, especially for SAMSum. However, this specialization comes at the cost of severely diminished performance on audio inputs. The audio fine-tuned model, `gemma_audio_fn`, did not demonstrate overall improvements and often underperformed compared to the base model, suggesting that the current audio fine-tuning approach may not be effective, probably because a train set bigger is needed to achieve better results.

### Results of fine-tuning varying the sampling rate of the processor

In [ ]:
SR_DIRS = {
    16000: GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO,
    24000: GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR24,
    8000:  GEMMA_RESULTS_DIR_FINE_TUNING_AUDIO_SR8,
}

MODALITIES = ["text", "audio"]
DECODINGS = ["greedy", "temperature", "beam"]

def build_config_from_dir(results_dir):

    cfg = {}
    for mod in MODALITIES:
        for dec in DECODINGS:
            fname = f"{mod}_summaries_{dec}.json"
            cfg[(mod, dec)] = os.path.join(results_dir, fname)
    return cfg

def compute_rows_for_config(config, sr_label, model_name="gemma_audio_fn", raise_on_missing=False):

    rows = []
    for (modality, decoding), path in config.items():
        if not os.path.exists(path):
            if raise_on_missing:
                raise FileNotFoundError(f"File mancante: {path}")
            else:
                # warn e skip
                print(f"Warn: file non trovato, skip: {path}")
                continue

        summary_key = f"summary_gemma_{modality}"
        data = load_predictions_with_dataset(path)
        pred_by_dataset = split_predictions_by_dataset(data, summary_key)

        rouge_xsum = get_rouge_scores(
            predictions=pred_by_dataset.get("xsum", []),
            references=gold_by_dataset["xsum"]
        )

        rouge_samsum = get_rouge_scores(
            predictions=pred_by_dataset.get("samsum", []),
            references=gold_by_dataset["samsum"]
        )

        rows.append({
            "model": model_name,
            "sr": sr_label,
            "modality": modality,
            "decoding": decoding,
            "xsum_rouge1": rouge_xsum["rouge1"],
            "xsum_rouge2": rouge_xsum["rouge2"],
            "xsum_rougeL": rouge_xsum["rougeL"],
            "xsum_rougeLsum": rouge_xsum["rougeLsum"],
            "samsum_rouge1": rouge_samsum["rouge1"],
            "samsum_rouge2": rouge_samsum["rouge2"],
            "samsum_rougeL": rouge_samsum["rougeL"],
            "samsum_rougeLsum": rouge_samsum["rougeLsum"],
        })

    return rows

all_rows = []
for sr, results_dir in SR_DIRS.items():
    cfg = build_config_from_dir(results_dir)
    rows = compute_rows_for_config(cfg, sr_label=sr, model_name="gemma_audio_fn")
    all_rows.extend(rows)

gemma_rouge_table_audio_fn_all = pd.DataFrame(all_rows)

cols_order = ["model", "sr", "modality", "decoding",
              "xsum_rouge1", "xsum_rouge2", "xsum_rougeL", "xsum_rougeLsum",
              "samsum_rouge1", "samsum_rouge2", "samsum_rougeL", "samsum_rougeLsum"]
gemma_rouge_table_audio_fn_all = gemma_rouge_table_audio_fn_all[cols_order]

In [ ]:
gemma_rouge_table_audio_fn_all

,model,sr,modality,decoding,xsum_rouge1,xsum_rouge2,xsum_rougeL,xsum_rougeLsum,samsum_rouge1,samsum_rouge2,samsum_rougeL,samsum_rougeLsum
0,gemma_audio_fn,16000,text,greedy,14.91,3.47,11.63,11.66,16.59,6.68,13.36,13.43
1,gemma_audio_fn,16000,text,temperature,16.38,3.77,12.46,12.47,18.32,7.50,14.97,15.08
2,gemma_audio_fn,16000,text,beam,21.55,4.72,16.55,16.52,29.81,12.16,23.91,23.95
3,gemma_audio_fn,16000,audio,greedy,12.13,0.69,8.87,8.87,11.35,0.42,8.09,8.11
4,gemma_audio_fn,16000,audio,temperature,13.49,0.94,10.69,10.72,9.71,0.36,7.71,7.71
5,gemma_audio_fn,16000,audio,beam,12.13,0.69,8.87,8.87,11.35,0.42,8.09,8.11
6,gemma_audio_fn,24000,text,greedy,14.84,3.21,10.88,10.89,16.53,7.11,13.59,13.56
7,gemma_audio_fn,24000,text,temperature,16.01,3.71,11.92,11.89,20.36,8.90,16.53,16.52
8,gemma_audio_fn,24000,text,beam,21.70,5.03,16.69,16.69,29.89,12.81,24.01,24.11
9,gemma_audio_fn,24000,audio,greedy,12.13,0.69,8.87,8.87,11.35,0.42,8.09,8.11


## Evaluation Summary

### 1. Effect of Decoding Strategy

Across all sampling rates (8k, 16k, 24k) and for both datasets (XSum, SAMSum):

- **Beam search consistently achieves the highest ROUGE scores**
- **Temperature sampling performs better than greedy**
- **Greedy decoding yields the lowest performance**

This pattern is stable for both:
- ROUGE-1
- ROUGE-2
- ROUGE-L
- ROUGE-Lsum

The improvement from greedy → beam is substantial, especially on **SAMSum**, where ROUGE-1 increases by ~10–13 points in text modality.



### 2. Effect of Sampling Rate (Text Modality)

For **text modality**, increasing the sampling rate improves performance:

Performance trend: 24000 Hz ≥ 16000 Hz > 8000 Hz

Example (Beam, XSum ROUGE-1):
- 8000 Hz → 20.66
- 16000 Hz → 21.55
- 24000 Hz → 21.70

The gains from 16k → 24k are marginal, while 8k shows consistent degradation.



### 3. Audio Modality Performance

In **audio modality**, results are:

- Significantly lower than text modality
- Almost identical across sampling rates
- Beam search provides no improvement over greedy

This suggests that is a lot harder to get good results with audio input after fine-tuning audio data.



### 4. Dataset Sensitivity

SAMSum shows:
- Larger gains from beam search
- Higher absolute ROUGE improvements compared to XSum

This may indicate that:
- Dialogue summarization benefits more from structured decoding
- Longer conversational context amplifies beam search advantages

